# Feed Forward Network with PyTorch

## AI Usage

I used AI to investigate and understand errors.

## PyTorch setup

In [220]:
import torch

In [221]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using device: {device}")

Using device: cuda


## Load data and create Datasets

In [222]:
from torchvision.datasets import MNIST
from torchvision import transforms

In [223]:
transform = transforms.ToTensor()
train_data = MNIST(root="data", train=True, download=True, transform=transform)
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [224]:
from torch.utils.data import Subset

Split the training data into training and validation sets

In [225]:
train_indices = list(range(50000))
val_indices = list(range(50000, len(train_data)))

val_data = Subset(train_data, val_indices)
train_data = Subset(train_data, train_indices)
print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")

Training set size: 50000
Validation set size: 10000


In [226]:
test_data = MNIST(root="data", train=False, download=True, transform=transform)
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: ToTensor()

## Defining the Neural Network

In [227]:
import torch.nn as nn

In [228]:
input_size = train_data[0][0].numel()

# I think to(device) is not needed here but I added it just in case
model1 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

model2 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

model3 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
).to(device)

## Defining the optimizer

In [229]:
import torch.optim as optim

In [230]:
optimizer1 = optim.Adam(model1.parameters(), lr=0.001)
optimizer2 = optim.Adam(model2.parameters(), lr=0.01)
optimizer3 = optim.SGD(model3.parameters(), lr=0.001)

## Defining loss function

In [231]:
criterion = nn.CrossEntropyLoss()

## Defining mini-batches creation function

In [232]:
import numpy as np

In [233]:
def create_minibatches(batch_size, x, y):
    total_data = len(x)
    indices = np.arange(total_data)

    for i in range(0, total_data, batch_size):
        batch_idx = indices[i:i + batch_size]
        x_batch = torch.stack([x[j] for j in batch_idx])
        y_batch = torch.tensor([y[j] for j in batch_idx], dtype=torch.long)
        yield x_batch, y_batch

## Train loop

In [234]:
from tqdm import tqdm

In [ ]:
def train(model, train_data, optimizer, criterion, epochs, batch_size):
    # Training data
    x_data = [x for x, _ in train_data]
    y_data = [y for _, y in train_data]

    # Validation data
    x_val = [x for x, y in val_data]
    y_val = [y for _, y in val_data]

    for epoch in range(epochs):
        # Train
        model.train()
        running_loss = 0.0
        train_correct = 0
        train_total = 0

        epoch_iterator = tqdm(
            create_minibatches(batch_size, x_data, y_data),
            total=(len(train_data) + batch_size - 1) // batch_size,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for inputs, labels in epoch_iterator:
            optimizer.zero_grad()
            # The dataset is using cpu, check if we are using gpu and move to gpu
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Accuracy
            preds = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

            running_loss += loss.item() * len(inputs)

        epoch_train_loss = running_loss / len(train_data)
        epoch_train_accuracy = train_correct / train_total

        # Eval
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in create_minibatches(batch_size, x_val, y_val):
                if inputs.device != device:
                    inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                # Accuracy
                preds = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

                val_loss += loss.item() * len(inputs)
        
        epoch_val_loss = val_loss / len(val_data)
        epoch_val_accuracy = val_correct / val_total

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}, Train Acc: {epoch_train_accuracy:.4f}, Val Acc: {epoch_val_accuracy:.4f}")

    return epoch_train_loss, epoch_val_loss, epoch_train_accuracy, epoch_val_accuracy

In [236]:
model1_trained = train(model1, train_data, optimizer1, criterion, epochs=30, batch_size=64)

Epoch 1/30: 100%|██████████| 782/782 [00:01<00:00, 391.53it/s]


Epoch [1/30], Loss: 0.3819, Val Loss: 0.1985


Epoch 2/30: 100%|██████████| 782/782 [00:01<00:00, 433.25it/s]


Epoch [2/30], Loss: 0.1828, Val Loss: 0.1428


Epoch 3/30: 100%|██████████| 782/782 [00:01<00:00, 424.66it/s]


...

Epoch [28/30], Loss: 0.0035, Val Loss: 0.1253


Epoch 29/30: 100%|██████████| 782/782 [00:01<00:00, 465.35it/s]


Epoch [29/30], Loss: 0.0020, Val Loss: 0.1307


Epoch 30/30: 100%|██████████| 782/782 [00:01<00:00, 468.91it/s]


Epoch [30/30], Loss: 0.0017, Val Loss: 0.1209


In [237]:
model2_trained = train(model2, train_data, optimizer2, criterion, epochs=20, batch_size=86)

Epoch 1/20: 100%|██████████| 582/582 [00:01<00:00, 334.36it/s]


Epoch [1/20], Loss: 0.2843, Val Loss: 0.1925


Epoch 2/20: 100%|██████████| 582/582 [00:01<00:00, 336.55it/s]


Epoch [2/20], Loss: 0.1604, Val Loss: 0.1411


Epoch 3/20: 100%|██████████| 582/582 [00:01<00:00, 367.07it/s]


...

Epoch [18/20], Loss: 0.0605, Val Loss: 0.2424


Epoch 19/20: 100%|██████████| 582/582 [00:01<00:00, 369.17it/s]


Epoch [19/20], Loss: 0.0614, Val Loss: 0.1840


Epoch 20/20: 100%|██████████| 582/582 [00:01<00:00, 307.19it/s]


Epoch [20/20], Loss: 0.0555, Val Loss: 0.2144


In [ ]:
model3_trained = train(model3, train_data, optimizer3, criterion, epochs=35, batch_size=32)

Epoch 1/35: 100%|██████████| 1563/1563 [00:03<00:00, 493.54it/s]


Epoch [1/35], Loss: 2.2910, Val Loss: 2.2688


Epoch 2/35: 100%|██████████| 1563/1563 [00:03<00:00, 505.37it/s]


Epoch [2/35], Loss: 2.2389, Val Loss: 2.1948


Epoch 3/35: 100%|██████████| 1563/1563 [00:02<00:00, 527.06it/s]


...

Epoch 25/35: 100%|██████████| 1563/1563 [00:03<00:00, 501.75it/s]


Epoch [25/35], Loss: 0.3381, Val Loss: 0.3113


Epoch 26/35: 100%|██████████| 1563/1563 [00:03<00:00, 463.08it/s]


Epoch [26/35], Loss: 0.3334, Val Loss: 0.3072


Epoch 27/35:  63%|██████▎   | 983/1563 [00:01<00:01, 514.57it/s]

In [ ]:
for model, (train_loss, val_loss, train_accuracy, val_accuracy) in zip(
    ["Model 1", "Model 2", "Model 3"],
    [model1_trained, model2_trained, model3_trained]
):
    print(f"{model} - Final Training Loss: {train_loss:.4f}, Final Validation Loss: {val_loss:.4f}, Training Accuracy: {train_accuracy:.4f}, Validation Accuracy: {val_accuracy:.4f}")

Model 1 - Final Training Loss: 0.0019, Final Validation Loss: 0.1528
Model 2 - Final Training Loss: 0.0499, Final Validation Loss: 0.1956
Model 3 - Final Training Loss: 0.2900, Final Validation Loss: 0.2687
